GFM1609 Thesis</br>
Extreme threshold calculation on daily precipitation

**Benny Istanto/KLI/G2501222008**</br>
Faculty of Mathematics and Natural Sciences</br>
Bogor Agricultural University

Adviser:</br>
1. Prof. Dr. Ir. Rizaldi Boer, M.S.
2. Dr. I Putu Santikayasa, S.Si, M.Sc.

--------------------------------------------------------

# IMERG Daily Precipitation Extreme Threshold

--------------------------------------------------------

## Data

There is only one data will be used in this analysis.

1. Satellite Rainfall Estimates: The Integrated Multi-satellitE Retrievals for GPM (IMERG) Final Run data were provided by the NASA/Goddard Space Flight Center's Precipitation Processing Center from their website at  https://gpm.nasa.gov/data/imerg

IMERG Final Run (IMERG-F) data originally available in netCDF format, 1-day data as 1 netCDF file.

All data and result from this analysis are available from this link: [to do]

## Configuration

Configuration is a crucial aspect of setting up any data analysis or processing workflow. Proper configuration ensures seamless access to data, efficient execution of tasks, and smooth integration of required tools and libraries. This article covers several essential subtopics related to configuration, such as connecting Google Drive to Colab, installing packages, importing libraries, and setting up working directories.

### Google Drive directory into Colab

Connecting Google Drive to Colab is a vital step when working with data stored in Google Drive. It allows you to access and manipulate files directly from your Colab notebook. To connect your Google Drive, you can use the `google.colab.drive` module to mount your drive, enabling seamless access to your files and folders.

**Notes**

This is only apply if you are working Colab

In [1]:
# Connect Google Drive directory into Colab
from google.colab import drive

# Unmount the drive
drive.flush_and_unmount()

# Mount the drive again
drive.mount('/content/drive')


Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


### Install packages

By default, `xarray` uses the `scipy` backend for NetCDF I/O operations, which has limited support for compression. To use advanced features like compression, we need to explicitly specify the `netCDF4` backend. Let's install `netCDF4` first before execute the script.

In [2]:
!pip install netcdf4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 39.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.6 MB/s eta 0:00:00


## Event Identification Threshold

Extreme rainfall detection uses event-identification thresholds ([Mazzoglio et al. 2019](https://doi.org/10.3390/rs11060677)), issuing alerts if accumulated rainfall surpasses a predefined threshold. The thresholds will develop at the pixel level for each month, integrating both percentile and GEV distribution methods for robustness. Percentile-based thresholds are straightforward and intuitive, while GEV distribution offers statistical rigor, particularly for rare events, following the WMO Guidelines on the Definition and Characterization of Extreme Weather and Climate Events ([WMO-No. 1310](https://library.wmo.int/idurl/4/58396), page 6-7.)

This hybrid approach leverages the strengths of both methods:
* **Moderate and Heavy Rainfall:** Percentile-based thresholds (e.g., P50 or 2-year return period, P80 or 5-year return period) will be used for defining moderate and heavy rainfall events, as these are more common, and the percentile method is straightforward and reliable.
* **Intense and Extreme Rainfall:** GEV-based thresholds will be used for defining more extreme events (e.g., 10-year return period, 25-year return period), where the statistical rigor of the GEV distribution aids in accurately estimating the rare extremes.

![Flowchart_Threshold](https://drive.google.com/uc?export=view&id=167ujSVfCU4hfwxN9zji-JD-uKHgayfVp)

### Step 1: Calculating the Maximum Value for Each Month

This script computes the maximum precipitation values for each month from the daily and multi-day accumulated datasets. It begins by checking for the existence of pre-combined yearly datasets. If these datasets do not exist, the script creates them and stores them in a shared directory for reuse. This ensures efficient processing and avoids redundant computations.

The script then loads the combined datasets, resamples the data to a monthly frequency, calculates the maximum precipitation values for each month, and saves the results in new files with appropriate naming conventions. This helps identify extreme precipitation events on a monthly basis across different accumulation periods. The output files are compressed to save storage space and follow CF conventions for metadata.

The main goal of the following code is to resample the daily precipitation data to a monthly frequency and then calculate the maximum daily precipitation for each month:

```python
# Resample to monthly frequency and calculate daily maximum by month
monthly_max = combined_ds.resample(time='MS').max(dim='time')
```

Here's a detailed explanation:

#### Resampling

**Resampling** is the process of changing the frequency of your time series data. You can either upsample (increase the frequency) or downsample (decrease the frequency).

#### 'MS' in Resampling

- `'MS'` stands for "Month Start." It is a date offset alias used in the pandas library (and xarray, which builds on pandas) that specifies that the resampling should be done with a monthly frequency, with each resampling period starting at the beginning of the month.
- In this context, `'MS'` groups the data into monthly periods, with each period starting on the first day of the month.

#### Calculating the Maximum

- After resampling, `.max(dim='time')` is used to compute the maximum value within each resampling period (i.e., within each month).
- The dimension specified, `dim='time'`, indicates that the operation should be performed along the time axis.

#### Combining It All

The code `monthly_max = combined_ds.resample(time='MS').max(dim='time')`:

1. **Resamples** the daily precipitation data to a monthly frequency (`'MS'`), grouping all days of each month together.
2. **Calculates the maximum** daily precipitation for each month from the grouped daily data.

By resampling and calculating the maximum daily precipitation for each month, the data is transformed from a daily time series into a monthly summary, highlighting the most extreme daily precipitation values within each month.

In [1]:
import xarray as xr
import os
import glob
from tqdm import tqdm
import pandas as pd

# Main directory on Google Drive
dir = f'/content/drive/MyDrive/exercises/gfm1609'

# Define directories and filenames for max value calculations
max_calculations = [
    {"days": 1, "input_dir": f'{dir}/data/precip/imerg/fdd_1day', "output_dir": f'{dir}/data/precip/imerg/fddmaxm_1day'},
    {"days": 2, "input_dir": f'{dir}/data/precip/imerg/fdd_2day', "output_dir": f'{dir}/data/precip/imerg/fddmaxm_2day'},
    {"days": 3, "input_dir": f'{dir}/data/precip/imerg/fdd_3day', "output_dir": f'{dir}/data/precip/imerg/fddmaxm_3day'},
    {"days": 4, "input_dir": f'{dir}/data/precip/imerg/fdd_4day', "output_dir": f'{dir}/data/precip/imerg/fddmaxm_4day'},
    {"days": 5, "input_dir": f'{dir}/data/precip/imerg/fdd_5day', "output_dir": f'{dir}/data/precip/imerg/fddmaxm_5day'},
]

# Directory to store combined yearly datasets
combined_year_dir = f'{dir}/data/precip/imerg/combined_year_ds'

# Global variable to store user's choice
user_choice = None

def set_user_decision():
    """Prompt user for decision on existing files and store it globally."""
    global user_choice
    if user_choice is None:
        decision = input("An output file already exists. Choose an action - Replace (R), Skip (S), Abort (A): ").upper()
        while decision not in ['R', 'S', 'A']:
            print("Invalid choice. Please choose again.")
            decision = input("Choose an action - Replace (R), Skip (S), Abort (A): ").upper()
        user_choice = decision

# Specify compression settings
comp = dict(zlib=True, complevel=5)

def create_combined_yearly_ds(input_dir, days):
    """Create combined yearly datasets and save them to disk."""
    # Check if input folder exists
    if not os.path.exists(input_dir):
        print(f"Input folder does not exist: {input_dir}")
        exit(1)

    os.makedirs(combined_year_dir, exist_ok=True)

    # Create a list of files
    file_list = sorted(glob.glob(os.path.join(input_dir, f"idn_cli_imerg_fdd_{days}day_*.nc4")))

    # Group files by year
    files_by_year = {}
    for file_path in file_list:
        date_str = file_path.split('_')[-1].split('.')[0]
        year = date_str[:4]
        if year not in files_by_year:
            files_by_year[year] = []
        files_by_year[year].append(file_path)

    # Process each year separately and create combined dataset for each year
    with tqdm(total=len(files_by_year), desc=f"Processing {days}-day data by year") as year_pbar:
        for year, year_files in files_by_year.items():
            combined_ds_path = os.path.join(combined_year_dir, f"combined_{days}day_{year}.nc4")
            if not os.path.exists(combined_ds_path):
                with tqdm(total=len(year_files), desc=f"Combining datasets for {year}", leave=False) as combine_pbar:
                    datasets = [xr.open_dataset(f) for f in year_files]
                    combined_year_ds = xr.concat(datasets, dim='time')
                    combine_pbar.update(len(year_files))

                    combined_year_ds.to_netcdf(combined_ds_path)
                    combined_year_ds.close()
                    for ds in datasets:
                        ds.close()
                print(f"Combined dataset saved to {combined_ds_path}")
            year_pbar.update(1)

for calc in max_calculations:
    days = calc["days"]
    input_dir = calc["input_dir"]
    output_dir = calc["output_dir"]

    os.makedirs(output_dir, exist_ok=True)

    # Create combined yearly datasets if they don't already exist
    create_combined_yearly_ds(input_dir, days)

    # Load and process each combined dataset by year
    with tqdm(total=len(os.listdir(combined_year_dir)), desc=f"Calculating monthly max for {days}-day data") as year_pbar:
        for combined_ds_file in os.listdir(combined_year_dir):
            if f"combined_{days}day_" not in combined_ds_file:
                continue
            combined_ds_path = os.path.join(combined_year_dir, combined_ds_file)
            combined_ds = xr.open_dataset(combined_ds_path)

            # Resample to monthly frequency and calculate daily maximum by month
            monthly_max = combined_ds.resample(time='MS').max(dim='time')

            # Save each month's maximum as a separate file
            for date, ds in tqdm(monthly_max.groupby('time', squeeze=False), desc=f"Saving {days}-day monthly max files", leave=False):
                year_month = pd.Timestamp(date.item()).strftime('%Y%m')
                output_filename = f"idn_cli_imerg_fddmax_{days}day_{year_month}.nc4"
                output_filepath = os.path.join(output_dir, output_filename)

                if os.path.exists(output_filepath):
                    if user_choice is None:
                        set_user_decision()
                    if user_choice == 'S':
                        print(f"Skipping existing file: {output_filepath}")
                        continue  # Skip to the next file
                    elif user_choice == 'A':
                        print("Aborting process.")
                        exit(0)  # Exit the script
                    elif user_choice == 'R':
                        pass  # Replace the file

                try:
                    # Ensure CF conventions and compression
                    encoding = {var: comp for var in ds.data_vars}
                    ds.attrs['Conventions'] = 'CF-1.8'
                    ds.to_netcdf(output_filepath, encoding=encoding, engine='netcdf4')
                    print(f"Saved: {output_filepath}")
                except Exception as e:
                    print(f"Error saving file {output_filepath}: {e}")

            print(f"{days}-day monthly max processing complete for {combined_ds_file}.")
            year_pbar.update(1)

print("All monthly max calculations processed.")


Saving 1-day monthly max files:   0%|          | 0/7 [00:00<?, ?it/s]

An output file already exists. Choose an action - Replace (R), Skip (S), Abort (A): R



Saving 1-day monthly max files:  57%|█████▋    | 4/7 [00:06<00:03,  1.25s/it]

Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/fddmaxm_1day/idn_cli_imerg_fddmax_1day_200006.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/fddmaxm_1day/idn_cli_imerg_fddmax_1day_200007.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/fddmaxm_1day/idn_cli_imerg_fddmax_1day_200008.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/fddmaxm_1day/idn_cli_imerg_fddmax_1day_200009.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/fddmaxm_1day/idn_cli_imerg_fddmax_1day_200010.nc4



Calculating monthly max for 1-day data:   1%|          | 1/120 [00:13<27:21, 13.80s/it]

...

Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/fddmaxm_5day/idn_cli_imerg_fddmax_5day_202310.nc4



Saving 5-day monthly max files:  92%|█████████▏| 11/12 [00:08<00:00,  1.34it/s]

Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/fddmaxm_5day/idn_cli_imerg_fddmax_5day_202311.nc4



Calculating monthly max for 5-day data:  20%|██        | 24/120 [05:25<21:40, 13.55s/it]

Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/fddmaxm_5day/idn_cli_imerg_fddmax_5day_202312.nc4
5-day monthly max processing complete for combined_5day_2023.nc4.
All monthly max calculations processed.


### Step 2: Combine NetCDF Files by Month

This script reads the NetCDF files, groups them by month, and combines the data for each month into a single NetCDF file. These monthly aggregated NetCDF files are then saved in a specified directory. This approach ensures that the data is well-organized and easily accessible for further analysis.

With the following script, the goal is to efficiently group and combine daily or multi-day accumulated NetCDF files by month and save the combined data into monthly NetCDF files:

Here's a detailed explanation:

#### Grouping by Month

The script groups NetCDF files by month based on their filenames. It extracts the month from the filenames and organizes the files accordingly.

#### Combining Datasets

The function `combine_datasets_in_batches` combines the datasets in batches to manage memory usage efficiently:

- **Batch Processing**: The files for each month are processed in batches to avoid memory issues. Each batch is concatenated along the time dimension to create a single dataset for the month.
- **Concatenation**: The datasets within each batch are concatenated using `xr.concat(datasets, dim="time")`.

#### Saving Combined Datasets

After combining the datasets for each month, the combined dataset is saved to a NetCDF file with CF conventions and compression:

- **Compression**: The output files are compressed to save storage space.
- **CF Conventions**: The combined datasets follow CF conventions for metadata.

By grouping and combining the daily or multi-day accumulated NetCDF files by month, the script ensures that the data is well-organized and easily accessible for further analysis.

In [1]:
import xarray as xr
import os
import glob
from tqdm import tqdm

# Main directory
dir = f'/content/drive/MyDrive/exercises/gfm1609'

# Define the input directories and combined output directory
threshold_calculations = [
    {"days": 1, "input_dir": f'{dir}/data/precip/imerg/fddmaxm_1day', "output_dir": f'{dir}/data/precip/imerg/combined_month_ds'},
    {"days": 2, "input_dir": f'{dir}/data/precip/imerg/fddmaxm_2day', "output_dir": f'{dir}/data/precip/imerg/combined_month_ds'},
    {"days": 3, "input_dir": f'{dir}/data/precip/imerg/fddmaxm_3day', "output_dir": f'{dir}/data/precip/imerg/combined_month_ds'},
    {"days": 4, "input_dir": f'{dir}/data/precip/imerg/fddmaxm_4day', "output_dir": f'{dir}/data/precip/imerg/combined_month_ds'},
    {"days": 5, "input_dir": f'{dir}/data/precip/imerg/fddmaxm_5day', "output_dir": f'{dir}/data/precip/imerg/combined_month_ds'},
]

# Global variable to store user's choice
user_choice = None

def set_user_decision():
    """
    Prompt user for decision on existing files and store it globally.

    This function prompts the user to decide whether to replace, skip, or abort
    if an output file already exists. The decision is stored in a global variable
    to be used throughout the script.
    """
    global user_choice
    if user_choice is None:
        decision = input("An output file already exists. Choose an action - Replace (R), Skip (S), Abort (A): ").upper()
        while decision not in ['R', 'S', 'A']:
            print("Invalid choice. Please choose again.")
            decision = input("Choose an action - Replace (R), Skip (S), Abort (A): ").upper()
        user_choice = decision

# Specify compression settings
comp = dict(zlib=True, complevel=5)

# Function to combine datasets in batches
def combine_datasets_in_batches(month_files, batch_size=10):
    batches = [month_files[i:i + batch_size] for i in range(0, len(month_files), batch_size)]
    combined_ds = None

    for batch in batches:
        datasets = []
        for file_path in batch:
            try:
                ds = xr.open_dataset(file_path)
                datasets.append(ds)
            except Exception as e:
                print(f"Error opening dataset {file_path}: {e}")
                continue

        if combined_ds is None:
            combined_ds = xr.concat(datasets, dim="time")
        else:
            combined_ds = xr.concat([combined_ds] + datasets, dim="time")

        # Close and delete individual datasets to free up memory
        for ds in datasets:
            ds.close()
            del ds

    return combined_ds

# Combine NetCDF files by month
for calc in threshold_calculations:
    days = calc["days"]
    input_dir = calc["input_dir"]
    output_dir = calc["output_dir"]

    os.makedirs(output_dir, exist_ok=True)

    file_list = sorted(glob.glob(os.path.join(input_dir, f"idn_cli_imerg_fddmax_{days}day_*.nc4")))

    files_by_month = {}
    for file_path in file_list:
        date_str = file_path.split('_')[-1].split('.')[0]
        month = date_str[4:6]
        if month not in files_by_month:
            files_by_month[month] = []
        files_by_month[month].append(file_path)

    for month, month_files in files_by_month.items():
        try:
            combined_ds = combine_datasets_in_batches(month_files, batch_size=10)

            combined_nc_filename = f"combined_{month}_{days}day.nc4"
            combined_nc_filepath = os.path.join(output_dir, combined_nc_filename)

            if os.path.exists(combined_nc_filepath):
                if user_choice is None:
                    set_user_decision()
                if user_choice == 'S':
                    print(f"Skipping existing file: {combined_nc_filepath}")
                    continue  # Skip to the next file
                elif user_choice == 'A':
                    print("Aborting process.")
                    exit(0)  # Exit the script
                elif user_choice == 'R':
                    pass  # Replace the file

            # Ensure CF conventions and compression
            encoding = {var: comp for var in combined_ds.data_vars}
            combined_ds.attrs['Conventions'] = 'CF-1.8'
            combined_ds.to_netcdf(combined_nc_filepath, encoding=encoding, engine='netcdf4')
            print(f"Saved: {combined_nc_filepath}")

            # Close and delete combined dataset to free up memory
            combined_ds.close()
            del combined_ds

        except Exception as e:
            print(f"Error combining datasets for month {month}: {e}")
            continue

print("All NetCDF files have been combined by month.")


Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/combined_month_ds/combined_06_1day.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/combined_month_ds/combined_07_1day.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/combined_month_ds/combined_08_1day.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/combined_month_ds/combined_09_1day.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/combined_month_ds/combined_10_1day.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/combined_month_ds/combined_11_1day.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/combined_month_ds/combined_12_1day.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/combined_month_ds/combined_01_1day.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/precip/imerg/combined_month_ds/combined_02_1day.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/pr

### Step 3: Calculate Threshold by Month

This script calculates the thresholds for extreme rainfall events using both percentile-based and Generalized Extreme Value (GEV) methods. It processes daily maximum precipitation data, grouped by month, and calculates the thresholds for moderate, heavy, intense, and extreme rainfall events. The thresholds are saved as NetCDF files, compressed to save storage space, and follow CF conventions for metadata. The script handles user decisions for existing files and provides progress tracking using tqdm. The output filenames reflect the accumulation period (1-5 days) and the rainfall event class (moderate, heavy, intense, extreme).

1. **Monthly Maxima**:
   - The script utilizes the monthly maxima defined by the previous script, which combines data for the same month across different years. This aggregation is crucial for fitting the GEV distribution, as it ensures the fitting is done on the most extreme values for each month.

2. **GEV and Percentile Calculation**:
   - The `calculate_percentile_threshold` function calculates the percentile threshold for each pixel's time series individually.
   - The `calculate_gev_threshold` function iterates over each pixel `(lat, lon)` and fits the GEV distribution to the non-NaN values at each pixel location, ensuring the fitting is done over time for each pixel.

3. **Handling NaN values**:
   - NaN values are removed from the data before fitting the GEV distribution. If a pixel has no valid values or insufficient variability, the fitting is skipped for that pixel.

This ensures that the fitting is correctly done on a pixel-by-pixel basis over time, rather than over the entire area.


In [3]:
import pandas as pd
import numpy as np
import os
import glob
from tqdm import tqdm
from scipy import optimize
from scipy.stats import genextreme
import xarray as xr

# Main directory on Google Drive
dir = f'/content/drive/MyDrive/exercises/gfm1609'

# Define directories for combined NetCDF files and threshold calculations
threshold_calculations = [
    {"days": 1, "combined_files_dir": f'{dir}/data/precip/imerg/combined_month_ds', "output_dir": f'{dir}/data/extremes/threshold_fddmaxm_1day'},
    {"days": 2, "combined_files_dir": f'{dir}/data/precip/imerg/combined_month_ds', "output_dir": f'{dir}/data/extremes/threshold_fddmaxm_2day'},
    {"days": 3, "combined_files_dir": f'{dir}/data/precip/imerg/combined_month_ds', "output_dir": f'{dir}/data/extremes/threshold_fddmaxm_3day'},
    {"days": 4, "combined_files_dir": f'{dir}/data/precip/imerg/combined_month_ds', "output_dir": f'{dir}/data/extremes/threshold_fddmaxm_4day'},
    {"days": 5, "combined_files_dir": f'{dir}/data/precip/imerg/combined_month_ds', "output_dir": f'{dir}/data/extremes/threshold_fddmaxm_5day'},
]

# Threshold definitions
threshold_definitions = [
    {"name": "moderate", "percentile": 50, "years": 2, "output_suffix": "02yr"},
    {"name": "heavy", "percentile": 80, "years": 5, "output_suffix": "05yr"},
    {"name": "intense", "percentile": None, "years": 10, "output_suffix": "10yr"},
    {"name": "extreme", "percentile": None, "years": 25, "output_suffix": "25yr"},
]

# Global variable to store user's choice
user_choice = None

def set_user_decision():
    """
    Prompt user for decision on existing files and store it globally.

    This function prompts the user to decide whether to replace, skip, or abort
    if an output file already exists. The decision is stored in a global variable
    to be used throughout the script.
    """
    global user_choice
    if user_choice is None:
        decision = input("An output file already exists. Choose an action - Replace (R), Skip (S), Abort (A): ").upper()
        while decision not in ['R', 'S', 'A']:
            print("Invalid choice. Please choose again.")
            decision = input("Choose an action - Replace (R), Skip (S), Abort (A): ").upper()
        user_choice = decision

def calculate_percentile_threshold(data, percentile):
    """
    Calculate the percentile threshold for the given data.

    Parameters:
    data (numpy.ndarray): The input data.
    percentile (float): The percentile to calculate (e.g., 50 for median).

    Returns:
    float: The calculated percentile value.
    """
    return np.nanpercentile(data, percentile)

def gev_fit_moments(data):
    """
    Fit the Generalized Extreme Value (GEV) distribution to the data using the method of moments.

    Parameters:
    data (numpy.ndarray): The input data.

    Returns:
    tuple: The shape, location, and scale parameters of the GEV distribution.
    """
    def moment_equations(params):
        shape, loc, scale = params
        if abs(shape) < 1e-8:
            return (loc + scale * np.euler_gamma - np.mean(data),
                    scale**2 * np.pi**2 / 6 - np.var(data),
                    0)
        else:
            return (loc + scale * (1 - np.euler_gamma * shape) / shape - np.mean(data),
                    scale**2 * (1 - 2 * shape * np.euler_gamma + shape**2 * np.pi**2 / 6) / shape**2 - np.var(data),
                    np.sign(shape) * (3 * np.euler_gamma * shape - 1 - shape**3 * np.pi**2 / 6) / (1 - 2 * shape) - np.sign(np.mean((data - np.mean(data))**3)))

    initial_guess = [0, np.mean(data), np.std(data)]
    shape, loc, scale = optimize.fsolve(moment_equations, initial_guess)
    return shape, loc, scale

def calculate_gev_threshold(data, return_period, lat, lon):
    """
    Calculate the Generalized Extreme Value (GEV) threshold for a given return period.

    Parameters:
    data (numpy.ndarray): The input data.
    return_period (int): The return period for which to calculate the threshold.
    lat (float): The latitude of the data point (for logging purposes).
    lon (float): The longitude of the data point (for logging purposes).

    Returns:
    float: The calculated GEV threshold.
    """
    data = data[~np.isnan(data)]  # Remove NaN values
    if data.size < 10:  # Require at least 10 valid data points
        print(f"Warning: Insufficient data for GEV calculation at lat {lat}, lon {lon}")
        return np.nan
    if np.nanstd(data) == 0:
        print(f"Warning: Zero standard deviation in data at lat {lat}, lon {lon}")
        return np.nan
    try:
        shape, loc, scale = gev_fit_moments(data)
        threshold = genextreme.ppf(1 - 1 / return_period, shape, loc=loc, scale=scale)
        if np.isnan(threshold) or np.isinf(threshold) or threshold > 1e4:  # Sanity check
            print(f"Warning: Unusual GEV result at lat {lat}, lon {lon}")
            print(f"Shape: {shape}, Loc: {loc}, Scale: {scale}, Threshold: {threshold}")
            print(f"Data: {data}")
            return np.nan
        return threshold
    except Exception as e:
        print(f"Error in GEV calculation at lat {lat}, lon {lon}: {str(e)}")
        print(f"Data: {data}")
        return np.nan

def remove_outliers(data, k=1.5):
    """
    Remove outliers from the data using the Interquartile Range (IQR) method.

    Parameters:
    data (numpy.ndarray): The input data.
    k (float): The IQR multiplier to determine the bounds for outliers. Default is 1.5.

    Returns:
    numpy.ndarray: The data with outliers removed.
    """
    q1, q3 = np.nanpercentile(data, [25, 75])
    iqr = q3 - q1
    lower_bound = q1 - k * iqr
    upper_bound = q3 + k * iqr
    return data[(data >= lower_bound) & (data <= upper_bound)]

def print_data_summary(data, lat, lon):
    print(f"Data summary for lat {lat}, lon {lon}:")
    print(f"  Min: {np.nanmin(data)}")
    print(f"  Max: {np.nanmax(data)}")
    print(f"  Mean: {np.nanmean(data)}")
    print(f"  Median: {np.nanmedian(data)}")
    print(f"  Std Dev: {np.nanstd(data)}")
    print(f"  Number of non-NaN values: {np.sum(~np.isnan(data))}")

# Perform threshold calculation on combined NetCDF files and save to NetCDF4
for calc in threshold_calculations:
    days = calc["days"]
    combined_files_dir = calc["combined_files_dir"]
    output_dir = calc["output_dir"]

    os.makedirs(output_dir, exist_ok=True)

    combined_file_list = sorted(glob.glob(os.path.join(combined_files_dir, f"combined_*_{days}day.nc4")))

    # At the start of your main loop (before the file processing begins):
    unusual_count = 0
    total_processed = 0

    for combined_file in tqdm(combined_file_list, desc=f"Calculating thresholds for {days}-day data"):
        month = combined_file.split('_')[-2]
        combined_ds = xr.open_dataset(combined_file)

        lats = combined_ds.lat.values
        lons = combined_ds.lon.values
        combined_data = combined_ds['precipitation'].values  # Extract precipitation data

        for threshold_def in threshold_definitions:
            name = threshold_def["name"]
            percentile = threshold_def["percentile"]
            years = threshold_def["years"]
            output_suffix = threshold_def["output_suffix"]

            output_filename_nc4 = f"idn_cli_imerg_fdd_{days}day_extreme_threshold_{output_suffix}_{month}.nc4"
            output_filepath_nc4 = os.path.join(output_dir, output_filename_nc4)

            if os.path.exists(output_filepath_nc4):
                if user_choice is None:
                    set_user_decision()
                if user_choice == 'S':
                    print(f"Skipping existing file: {output_filepath_nc4}")
                    continue  # Skip to the next file
                elif user_choice == 'A':
                    print("Aborting process.")
                    exit(0)  # Exit the script
                elif user_choice == 'R':
                    pass  # Replace the file

            thresholds = np.full((len(lats), len(lons)), np.nan)  # Initialize threshold array

            for i in range(len(lats)):
                for j in range(len(lons)):
                    lat, lon = lats[i], lons[j]
                    data = combined_data[:, j, i]  # Adjust the indexing order for correct dimensions
                    if np.all(np.isnan(data)):  # Skip rows with all NaN values
                        continue

                    # Handle outliers
                    data = remove_outliers(data)

                    total_processed += 1

                    if percentile is not None:
                        threshold = calculate_percentile_threshold(data, percentile)
                    else:
                        threshold = calculate_gev_threshold(data, years, lat, lon)

                    if np.isnan(threshold) or np.isinf(threshold) or threshold > 1e10:  # Add a sanity check
                        unusual_count += 1
                        print(f"\nWarning: Unusual threshold {threshold} for lat {lat}, lon {lon}")
                        print_data_summary(data, lat, lon)
                        threshold = np.nan  # Set to NaN if the result is unreasonable

                    thresholds[i, j] = threshold

            # Create a Dataset to store the thresholds
            ds_threshold = xr.Dataset(
                {
                    'threshold': (['lat', 'lon'], thresholds)
                },
                coords={
                    'lat': (['lat'], lats),
                    'lon': (['lon'], lons)
                }
            )

            # Add metadata
            ds_threshold.attrs['Conventions'] = 'CF-1.8'
            ds_threshold['threshold'].attrs['long_name'] = f'{days}-day {name} or {output_suffix} precipitation threshold'
            ds_threshold['threshold'].attrs['units'] = 'mm'
            ds_threshold['lat'].attrs['units'] = 'degrees_north'
            ds_threshold['lat'].attrs['long_name'] = 'Latitude'
            ds_threshold['lon'].attrs['units'] = 'degrees_east'
            ds_threshold['lon'].attrs['long_name'] = 'Longitude'

            # Save the Dataset to NetCDF with compression
            comp = dict(zlib=True, complevel=5)
            encoding = {var: comp for var in ds_threshold.data_vars}
            ds_threshold.to_netcdf(output_filepath_nc4, encoding=encoding, engine='netcdf4')
            print(f"Saved: {output_filepath_nc4}")

    # After processing all files for this calculation (moved outside the inner loops):
    print(f"\nProcessed {total_processed} rows.")
    print(f"Found {unusual_count} unusual results.")
    print(f"Percentage of unusual results: {unusual_count/total_processed*100:.2f}%")

print("All thresholds have been calculated.")



Calculating thresholds for 1-day data:   0%|          | 0/12 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_1day/idn_cli_imerg_fdd_1day_extreme_threshold_02yr_01.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_1day/idn_cli_imerg_fdd_1day_extreme_threshold_05yr_01.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_1day/idn_cli_imerg_fdd_1day_extreme_threshold_10yr_01.nc4


Calculating thresholds for 1-day data:   8%|▊         | 1/12 [01:09<12:48, 69.84s/it]

Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_1day/idn_cli_imerg_fdd_1day_extreme_threshold_25yr_01.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_1day/idn_cli_imerg_fdd_1day_extreme_threshold_02yr_02.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_1day/idn_cli_imerg_fdd_1day_extreme_threshold_05yr_02.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_1day/idn_cli_imerg_fdd_1day_extreme_threshold_10yr_02.nc4


Calculating thresholds for 1-day data:  17%|█▋        | 2/12 [02:20<11:41, 70.11s/it]

...

Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_5day/idn_cli_imerg_fdd_5day_extreme_threshold_25yr_10.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_5day/idn_cli_imerg_fdd_5day_extreme_threshold_02yr_11.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_5day/idn_cli_imerg_fdd_5day_extreme_threshold_05yr_11.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_5day/idn_cli_imerg_fdd_5day_extreme_threshold_10yr_11.nc4


Calculating thresholds for 5-day data:  92%|█████████▏| 11/12 [13:08<01:11, 71.19s/it]

Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_5day/idn_cli_imerg_fdd_5day_extreme_threshold_25yr_11.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_5day/idn_cli_imerg_fdd_5day_extreme_threshold_02yr_12.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_5day/idn_cli_imerg_fdd_5day_extreme_threshold_05yr_12.nc4
Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_5day/idn_cli_imerg_fdd_5day_extreme_threshold_10yr_12.nc4


Calculating thresholds for 5-day data: 100%|██████████| 12/12 [14:19<00:00, 71.60s/it]

Saved: /content/drive/MyDrive/exercises/gfm1609/data/extremes/threshold_fddmaxm_5day/idn_cli_imerg_fdd_5day_extreme_threshold_25yr_12.nc4

Processed 930864 rows.
Found 0 unusual results.
Percentage of unusual results: 0.00%
All thresholds have been calculated.


End of script.